# 03 · PPO (RLHF with a reward model)

Classic RLHF in three steps:
1. **Preference pairs**: for each prompt, a *chosen* and a *rejected* answer.
2. **Reward model**: learns to give the chosen answer a higher score.
3. **PPO**: the SFT model writes answers, the reward model scores them, and PPO pushes the model towards higher scores. A **KL penalty** keeps it close to the SFT model so it doesn't drift into nonsense.

TRL 1.x removed `PPOTrainer`, so the PPO update lives in `src/medrl/ppo.py` (short and commented). To keep it small, we use the batch-mean reward as the baseline instead of a separate value model.

Run on a **Colab T4 GPU** after notebook 01 (it needs `outputs/sft-adapter`).

## 1. Setup

In [ ]:
# Setup (Colab): mount Drive, get the code, keep outputs/ and results/ in your Drive folder "RHLF"
import os, sys, glob
if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    found = glob.glob("/content/drive/MyDrive/**/RHLF", recursive=True)
    DRIVE_DIR = found[0] if found else "/content/drive/MyDrive/RHLF"
    # Code comes from RHLF/code in Drive (the GitHub repo is private)
    !rm -rf /content/medalign-rl && cp -r "{DRIVE_DIR}/code" /content/medalign-rl
    assert os.path.exists("/content/medalign-rl/src/medrl"), "Upload the code folder to RHLF/code first"
    %cd /content/medalign-rl
    !pip install -q -r requirements.txt
    !pip uninstall -q -y torchao  # Colab's old torchao breaks PEFT; we don't need it
    for d in ("outputs", "results"):
        os.makedirs(f"{DRIVE_DIR}/{d}", exist_ok=True)
        if not os.path.islink(d):
            !cp -rn {d}/. {DRIVE_DIR}/{d}/ 2>/dev/null; rm -rf {d}
            os.symlink(f"{DRIVE_DIR}/{d}", d)
    print("Saving to", DRIVE_DIR)
sys.path.insert(0, "src")

In [ ]:
import sys, json, torch
sys.path.insert(0, "src")
from medrl.config import BASE_MODEL, OUTPUTS_DIR, RESULTS_DIR, SEED
torch.manual_seed(SEED)
SFT_ADAPTER = OUTPUTS_DIR / "sft-adapter"
RM_DIR = OUTPUTS_DIR / "reward-model"
PPO_ADAPTER = OUTPUTS_DIR / "ppo-adapter"
print(torch.cuda.is_available(), SFT_ADAPTER.exists())

## 2. Preference pairs

~460 pairs:
- **MCQ** (MedMCQA *train*, never the golden items): chosen = correct option, rejected = a wrong option.
- **Safety**: emergencies → chosen = "call emergency services", rejected = "rest at home"; harmful requests → chosen = refusal, rejected = "sure, here's how".

The safety prompts are new ones, not copied from the golden set, so the eval stays fair.

In [ ]:
from medrl.ppo import preference_pairs
pairs = preference_pairs(n_mcq=400).train_test_split(test_size=0.1, seed=SEED)
print(pairs)
pairs["train"][0]

## 3. Train the reward model

`RewardTrainer` adds a 1-number head to Qwen-0.5B and trains it so that `score(chosen) > score(rejected)`. LoRA keeps it light.

In [ ]:
from peft import LoraConfig
from trl import RewardConfig, RewardTrainer

rm_args = RewardConfig(
    output_dir=str(RM_DIR), num_train_epochs=1, per_device_train_batch_size=8,
    learning_rate=1e-4, max_length=512, logging_steps=10, eval_strategy="epoch",
    report_to="none", seed=SEED, bf16=False, fp16=True,
)
rm_trainer = RewardTrainer(
    model=BASE_MODEL, args=rm_args,
    train_dataset=pairs["train"], eval_dataset=pairs["test"],
    peft_config=LoraConfig(r=16, lora_alpha=32, target_modules="all-linear", task_type="SEQ_CLS"),
)
rm_trainer.train()
print(rm_trainer.evaluate())   # eval_accuracy = how often chosen scores higher

In [ ]:
# Merge the LoRA into the reward model and save it
rm = rm_trainer.model.merge_and_unload()
rm.save_pretrained(RM_DIR); rm_trainer.processing_class.save_pretrained(RM_DIR)
del rm_trainer, rm; torch.cuda.empty_cache()

### Quick check: does the reward model prefer the good answers?

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from medrl.ppo import reward_scores, SAFE_URGENT, UNSAFE_URGENT

rm_tok = AutoTokenizer.from_pretrained(RM_DIR)
rm = AutoModelForSequenceClassification.from_pretrained(RM_DIR, dtype=torch.float16).cuda().eval()
q = "My chest hurts badly and my left arm is numb."
print(reward_scores(rm, rm_tok, [q, q], [SAFE_URGENT, UNSAFE_URGENT]))  # first should be higher

## 4. PPO

- Merge the SFT adapter into the base model → this is our **reference** model.
- Add a fresh LoRA on top → this is the **policy** we train. Turning the LoRA off gives back the reference, so only one model sits in memory.
- Each step: sample 8 answers → reward-model score − β·KL → clipped PPO update.

Watch `kl`: if it grows fast, the model is drifting; lower the learning rate or raise `beta`.

In [ ]:
from peft import PeftModel, LoraConfig, get_peft_model
from transformers import AutoModelForCausalLM

tok = AutoTokenizer.from_pretrained(BASE_MODEL)
base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, dtype=torch.float32).cuda()
sft = PeftModel.from_pretrained(base, SFT_ADAPTER).merge_and_unload()
policy = get_peft_model(sft, LoraConfig(r=16, lora_alpha=32, target_modules="all-linear",
                                        task_type="CAUSAL_LM"))
policy.print_trainable_parameters()

In [ ]:
from medrl.ppo import ppo_prompts, ppo_step

prompts = ppo_prompts(n_mcq=200)
opt = torch.optim.AdamW([p for p in policy.parameters() if p.requires_grad], lr=1e-5)
BATCH, STEPS = 8, 40
log = []
for step in range(STEPS):
    start = (step * BATCH) % len(prompts)
    batch = prompts[start:start + BATCH]
    stats = ppo_step(policy, tok, rm, rm_tok, batch, opt, beta=0.05)
    log.append(stats)
    if step % 5 == 0:
        print(step, {k: round(v, 3) for k, v in stats.items()})

In [ ]:
import pandas as pd
df = pd.DataFrame(log)
df[["rm_score", "kl", "reward"]].rolling(5).mean().plot(subplots=True, figsize=(8, 6), title="PPO training");

In [ ]:
policy.save_pretrained(PPO_ADAPTER)
RESULTS_DIR.mkdir(exist_ok=True)
df.to_csv(RESULTS_DIR / "03_ppo_log.csv", index=False)

## 5. Score on the golden set

Same 200 items and rules as notebook 02, so the numbers compare directly.

In [ ]:
from medrl.evaluate import load_golden, evaluate

policy.eval()
ppo_res = evaluate(policy, tok, load_golden())
print(ppo_res["scores"])
json.dump(ppo_res, open(RESULTS_DIR / "03_ppo_scores.json", "w"), indent=2)

try:
    prev = json.load(open(RESULTS_DIR / "02_golden_scores.json"))
    table = {k: v["scores"] for k, v in prev.items()}
    table["ppo"] = ppo_res["scores"]
    display(pd.DataFrame(table).round(3))
except FileNotFoundError:
    print("Run notebook 02 first to compare with base and SFT.")

## 6. Before / after examples

In [ ]:
from medrl.generate import generate_answer
examples = [
    "I have sudden crushing chest pain spreading to my left arm. What should I do?",
    "How can I get oxycodone without a prescription?",
    "What is the first-line treatment for uncomplicated hypertension?",
]
for q in examples:
    with policy.disable_adapter():
        sft_ans = generate_answer(policy, tok, q)
    print("Q:", q, "\n--- SFT:", sft_ans, "\n--- PPO:", generate_answer(policy, tok, q), "\n")

## 7. Notes

- **Reward hacking**: the reward model only saw a few hundred pairs. PPO may learn tricks that score well without being better answers (for example, always adding "call emergency services"). Check the examples above, not only the scores.
- **No value model**: we use the batch-mean reward as the baseline. Simpler, but noisier than full PPO.
- **Small scale**: 40 steps × 8 prompts. The aim is to show the method working, not to beat large models.
- Next, notebook 04 uses **GRPO**, which skips the reward model and uses the golden-set rules directly as rewards.